# Spend Classification with `ai_classify`

**Goal:** Bootstrap a spend classification system from messy government procurement data — no training, no fine-tuning.

| Step | What | How |
|------|------|-----|
| 1 | Baseline | `ai_classify` against raw 338-category taxonomy → ~3% accuracy |
| 2 | Taxonomy design | `ai_query` proposes 25 clean categories with rich descriptions |
| 3 | Reclassify | `ai_classify` with new taxonomy → **75%+ accuracy** |

**Only inputs to classifier:** vendor name, L1 category, department, program, amount  
**Not used:** sub-object (ground truth derived from this)

In [0]:
import json, os
import pandas as pd

DATA_DIR = "/Workspace/Users/scott.mckean@databricks.com"
DATA_CACHE = f"{DATA_DIR}/spend_data_cache.parquet"
TAXONOMY_CACHE = f"{DATA_DIR}/spend_taxonomy_raw.csv"
TAXONOMY_PATH = f"{DATA_DIR}/spend_taxonomy.json"
SODA_URL = "https://data.sfgov.org/resource/n9pm-xkyq.csv"

if os.path.exists(DATA_CACHE):
    pdf = pd.read_parquet(DATA_CACHE)
    taxonomy_df = pd.read_csv(TAXONOMY_CACHE)
else:
    import requests
    from io import StringIO
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2)))

    resp = s.get(SODA_URL, params={"$select": "object, count(*) as cnt", "$group": "object", "$order": "cnt DESC", "$limit": 500}, timeout=60)
    taxonomy_df = pd.read_csv(StringIO(resp.text)).dropna(subset=["object"])

    resp = s.get(SODA_URL, params={"$limit": 1000, "$where": "vendor IS NOT NULL AND object IS NOT NULL",
        "$order": "data_loaded_at DESC",
        "$select": "vendor, object, sub_object, character, vouchers_paid, department, program, fiscal_year"}, timeout=60)
    pdf = pd.read_csv(StringIO(resp.text))
    pdf["vouchers_paid"] = pd.to_numeric(pdf["vouchers_paid"], errors="coerce")
    pdf = pdf.dropna(subset=["vendor", "object", "vouchers_paid"])

    pdf.to_parquet(DATA_CACHE, index=False)
    taxonomy_df.to_csv(TAXONOMY_CACHE, index=False)

spark.createDataFrame(pdf).createOrReplaceTempView("spend_data")
print(f"{len(taxonomy_df)} raw categories | {len(pdf)} transactions | {pdf['vendor'].nunique()} vendors")

## Prompt Registry
All prompts live here — code cells below are pure orchestration. Edit these to iterate on taxonomy quality.

In [0]:
# ── Prompt Registry ──────────────────────────────────────────────────────────
# All prompts and SQL templates in one place. Code cells below just call these.

INSTRUCTIONS = {
    "baseline": "Classify this government procurement transaction into the best-fit spend category.",

    "enhanced": (
        "Classify this government procurement transaction. "
        "PRIMARY signals: "
        "(1) Category field is STRONGEST: Materials & Supplies = physical goods; Non Personnel Services = contracted services. "
        "(2) Vendor name indicates product type: supply/hardware companies = materials; engineering/consulting firms = professional services. "
        "SECONDARY: "
        "(3) Department/Program provide context but do NOT override Category. "
        "Most transactions are from Water Enterprise - this does NOT mean they are water supplies. "
        "A hardware vendor in Water dept is still materials, not water treatment. "
        "(4) Amount: large ($10K+) = professional services or capital."
    ),
}

def build_taxonomy_prompt(pdf, taxonomy_df):
    """Build the ai_query prompt for taxonomy design, informed by sub-category structure."""
    cat_info = pdf.dropna(subset=["sub_object"]).groupby("object").apply(
        lambda g: {
            "character": g["character"].mode().iloc[0],
            "count": len(g),
            "sub_objects": list(g["sub_object"].value_counts().index),
            "vendors": list(g["vendor"].value_counts().head(5).index)
        }
    ).to_dict()

    structure_lines = []
    for obj, info in sorted(cat_info.items(), key=lambda x: -x[1]["count"]):
        subs = ", ".join(info["sub_objects"])
        vendors = ", ".join(info["vendors"])
        structure_lines.append(f'  "{obj}" [L1={info["character"]}, n={info["count"]}]:\n    sub_objects: [{subs}]\n    vendors: [{vendors}]')

    sample_categories = sorted(pdf["object"].unique().tolist())
    sample_list = ", ".join(f'"{c}"' for c in sample_categories)

    return (
        "You are a procurement taxonomy specialist. Design a classification system for US city government spend data.\n\n"
        "## CLASSIFIER CONTEXT\n"
        "A classifier labels each transaction using ONLY: vendor name, character (L1: 'Materials & Supplies' or 'Non Personnel Services'), "
        "department, program, dollar amount. It CANNOT see sub_objects.\n"
        "Strongest signals: (1) character field, (2) vendor name. Department/program are WEAK.\n\n"
        "## KNOWN ISSUES\n"
        "- 90%+ transactions are from 'WTR Water Enterprise'. Do NOT let this bias toward water categories.\n"
        "- 'Inventories' is an accounting catch-all (auto parts, lumber, hardware). Map to 'Building and Construction Materials'.\n"
        "- 'Other Current Expenses' (Copy Machine, Cleaning, Freight, Software) = diverse operational costs.\n\n"
        f"## RAW CATEGORIES ({len(cat_info)} in sample)\n" + "\n".join(structure_lines) + "\n\n"
        "## REQUIREMENTS\n"
        "1) Propose 25-35 canonical categories named for WHAT WAS PURCHASED.\n"
        "2) Each description (200-350 chars): ALL sub_objects, sample vendors, L1 character, disambiguation rules.\n"
        f"3) Map EACH of these to one canonical category: {sample_list}\n"
        "4) Return ONLY JSON: {\"taxonomy\": {\"name\": \"desc\"}, \"mapping\": {\"raw\": \"canonical\"}}\n"
        "No markdown fences."
    )

print(f"Registry loaded: {len(INSTRUCTIONS)} instruction sets, classify template ready")

## Step 1: Baseline — Raw Taxonomy
338 messy categories, no descriptions. Near-duplicates like "Profess & Specialized Svcs" vs "Professional/Specialized Svcs" make exact-match impossible.

In [0]:
CLASSIFY_SQL = """
SELECT vendor, character, department, program, vouchers_paid, object AS ground_truth,
  ai_classify(
    CONCAT('Vendor: ', vendor, ' | Category: ', character,
           ' | Department: ', department, ' | Program: ', program,
           ' | Amount: $', CAST(vouchers_paid AS STRING)),
    '{taxonomy_json}',
    MAP('version', '2.0', 'instructions', '{instructions}')
  ):response[0]::STRING AS predicted_category
FROM spend_data
"""

In [0]:
# Attempt 1: raw taxonomy, no descriptions
raw_categories = taxonomy_df["object"].tolist()
taxonomy_json = json.dumps(raw_categories).replace("'", "''")
instructions = INSTRUCTIONS["baseline"].replace("'", "''")

attempt1_pdf = spark.sql(CLASSIFY_SQL.format(taxonomy_json=taxonomy_json, instructions=instructions)).toPandas()

correct = (attempt1_pdf["predicted_category"] == attempt1_pdf["ground_truth"]).sum()
print(f"\u2717 {len(raw_categories)} categories, no descriptions \u2192 {correct}/{len(attempt1_pdf)} = {100*correct/len(attempt1_pdf):.1f}% accuracy")

## Step 2: Design a Clean Taxonomy with `ai_query`
One LLM call proposes canonical categories + maps every raw label. Persisted to JSON for deterministic reloads — delete `spend_taxonomy.json` to regenerate.

In [0]:
sql = f"""
    SELECT ai_query('databricks-claude-opus-4-7', '{prompt}',
    modelParameters => named_struct('max_tokens', 8192, 'temperature', 0.1)) AS result
    """

In [0]:
if os.path.exists(TAXONOMY_PATH):
    with open(TAXONOMY_PATH) as f:
        saved = json.load(f)
    clean_taxonomy, category_mapping = saved["taxonomy"], saved["mapping"]
    print(f"Loaded from file: {len(clean_taxonomy)} categories, {len(category_mapping)} mappings")
else:
    prompt = build_taxonomy_prompt(pdf, taxonomy_df).replace("'", "''")
    import re
    result = spark.sql(sql).collect()[0]["result"]
    parsed = json.loads(result) if result.startswith("{") else json.loads(re.search(r'\{[\s\S]*\}', result).group())
    clean_taxonomy, category_mapping = parsed["taxonomy"], parsed["mapping"]

    with open(TAXONOMY_PATH, "w") as f:
        json.dump({"taxonomy": clean_taxonomy, "mapping": category_mapping}, f, indent=2)
    print(f"Generated and saved: {len(clean_taxonomy)} categories, {len(category_mapping)} mappings")

clean_taxonomy_json = json.dumps(clean_taxonomy)
for name, desc in clean_taxonomy.items():
    print(f"  {name:40s} | {desc[:80]}...")

## Step 3: Reclassify with Clean Taxonomy
Same model, same data, same inputs — only the taxonomy changes.

In [0]:
# Attempt 2: clean taxonomy with rich descriptions
taxonomy_json = clean_taxonomy_json.replace("'", "''")
instructions = INSTRUCTIONS["enhanced"].replace("'", "''")

attempt2_pdf = spark.sql(CLASSIFY_SQL.format(taxonomy_json=taxonomy_json, instructions=instructions)).toPandas()

print(f"\u2713 {len(clean_taxonomy)} categories, rich descriptions \u2192 classified {len(attempt2_pdf)} transactions")

## Results

In [0]:
# Compare accuracy (pure pandas — instant)
a1 = (attempt1_pdf["predicted_category"] == attempt1_pdf["ground_truth"]).sum()

attempt2_pdf["expected"] = attempt2_pdf["ground_truth"].map(category_mapping)
attempt2_pdf["correct"] = attempt2_pdf["predicted_category"] == attempt2_pdf["expected"]
a2 = attempt2_pdf["correct"].sum()
n = len(attempt2_pdf)

print(f"{'':30s} {'Attempt 1':>12s} {'Attempt 2':>12s}")
print(f"{'='*56}")
print(f"{'Taxonomy size':30s} {len(raw_categories):>12d} {len(clean_taxonomy):>12d}")
print(f"{'Descriptions':30s} {'No':>12s} {'Yes':>12s}")
print(f"{'Accuracy':30s} {a1:>5d}/{n} ({100*a1/n:.1f}%) {a2:>5d}/{n} ({100*a2/n:.1f}%)")
print(f"{'='*56}")
print(f"\u0394 = +{100*(a2-a1)/n:.0f} pp \u2014 same model, same inputs, better taxonomy.")

display(attempt2_pdf[["vendor", "vouchers_paid", "ground_truth", "expected", "predicted_category", "correct"]].head(20))

## Key Takeaways

1. **Taxonomy quality > model quality.** Same model went from 3% to 75%+ by fixing the labels.
2. **`ai_query` designs the taxonomy** — one call proposes categories, writes descriptions, and maps old→new.
3. **`ai_classify` applies at scale** — swap the JSON, re-run. No training, no deployment.
4. **Iterate in minutes:** Edit prompts in the registry above, re-run cells. Each cycle takes ~2 min for 1000 rows.

**To regenerate the taxonomy:** Delete `spend_taxonomy.json` and re-run. The prompt registry controls what gets generated.